# 07 — Dense embeddings (F2LLM-v2-0.6B, code-specialized)

Single-vector **dense** retrieval with **codefuse-ai/F2LLM-v2-0.6B** (1024-dim) — a **code-specialized** 0.6B LLM embedder served via **sentence-transformers** (torch). It's *asymmetric*: it applies its own built-in query-instruction prompt to queries; documents are embedded plain.

**Requires** `pip install -e '.[sentence-transformers]'` (already in the `.venv-li` kernel). Its embedder differs from bge-small, so this method uses its **own** index (own cache dir).

> Embeds with a 0.6B model — a few minutes on CPU. Add `--gpu` to the index command for CUDA.

### Index (slower — 0.6B model)

In [ ]:
import sys
# Index sample_repo for this method (CLI does the heavy ingestion wiring).
# --no-inspect = read source statically (don't import the package).
!{sys.executable} -m pydocs_mcp --config configs/dense_f2llm.yaml index sample_repo \
    --cache-dir .pydocs-cache/dense_f2llm --force --no-inspect 2>&1 \
    | grep -E "Project:|Done:|rror" || true

### Search (Python pipeline)

In [ ]:
from nb_helpers import make_searcher, load_queries, show_results

# Build the retrieval PIPELINE in Python for this method (no CLI search).
search = make_searcher("configs/dense_f2llm.yaml", cache_dir='.pydocs-cache/dense_f2llm')
queries = load_queries()

In [ ]:
for q in queries:
    hits = await search(q['query'], limit=5)
    show_results(q, hits)

### Takeaway
Code-tuned embeddings often beat general models on code retrieval — compare F2LLM's hits against bge-small / Qwen3 / gte-modernbert on the same queries.